In [95]:
import pandas as pd
import plotly.express as px
import streamlit_authenticator as stauth
import urllib
import mysql.connector
from sqlalchemy import create_engine, text

In [96]:
## Database connection
## Connection settings
server = 'sportsciencefb.database.windows.net'
database = 'sportsciencefb'
database_username = 'um_sport_science'
database_password = 'MiamiF00tball!'

## Connection Build to Azure
params = urllib.parse.quote_plus(
    f"DRIVER=ODBC Driver 18 for SQL Server;"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={database_username};"
    f"PWD={database_password};"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "Connection Timeout=30;"
    "MARS_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [97]:
get_players_sql_string = '''
select playerid, Player, positionID, app_username, app_password
from player p 
'''

In [98]:
get_players_sql_query = text(get_players_sql_string)
with engine.connect() as conn:
    player_df = pd.read_sql(get_players_sql_query, conn)
    player_df = player_df.dropna()
    names = player_df['Player'].to_list()
    usernames = player_df['app_username'].tolist()
    passwords = player_df['app_password'].tolist()
    playerids = player_df['playerid'].tolist()
    positionids = player_df['positionID'].tolist()

In [99]:
credentials = {"usernames": {}}
for name, playerid, positionid, username, password in zip(names, playerids, positionids, usernames, passwords):
    credentials["usernames"][username] = {
        "name": name,
        "playerID" : playerid,
        "positionID" : positionid,
        "password": password
    }

In [100]:
username = 'samsonokunlola'

In [101]:
playerid = player_df.loc[player_df['app_username'] == username]['playerid'].iloc[0]
positionid = player_df.loc[player_df['app_username'] == username]['positionID'].iloc[0]
print(playerid, positionid)

86 5.0


In [34]:
hashed_passwords = stauth.Hasher.hash_passwords(credentials=credentials)

In [35]:
credentials
import csv
with open("hashed_credentials.csv", "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    # Header
    writer.writerow(["username", "name", "playerID", "password"])
    
    # Rows
    for username, details in credentials["usernames"].items():
        writer.writerow([username, details["name"], details["playerID"], details["password"]])


In [102]:
get_cmj_sql_string = f'''
select playerID, date, [Jump Height]
from cmj
where playerID = {playerid}
'''

get_cmj_sql_query = text(get_cmj_sql_string)
with engine.connect() as conn:
    cmj_df = pd.read_sql(get_cmj_sql_query, conn)
cmj_df = cmj_df.sort_values(['date'])
player_peak_cmj = cmj_df.sort_values(['Jump Height']).tail(1)
player_peak_cmj_jump = player_peak_cmj['Jump Height'].iloc[0]
player_peak_cmj_date = player_peak_cmj['date'].iloc[0]
cmj_df = cmj_df.groupby(['date']).mean().reset_index()
cmj_df

,date,playerID,Jump Height
0,2023-03-20,86.0,0.369200
1,2023-03-24,86.0,0.401400
2,2023-03-27,86.0,0.372500
3,2023-03-31,86.0,0.398650
4,2023-04-03,86.0,0.381700
...,...,...,...
89,2025-08-01,86.0,0.351800
90,2025-08-08,86.0,0.382975
91,2025-08-22,86.0,0.507250
92,2025-08-26,86.0,0.240450


In [103]:
## Position Force Plate Data (2025)
get_position_cmj_sql_string = f'''
select p.playerID, pos.[Position], c.[date], c.[Jump Height]
from cmj c
join player p on c.playerID=p.playerid 
JOIN positions pos ON p.positionID = pos.positionID
where p.positionID = {positionid} AND c.[date] > '2025-01-01'
'''
get_position_cmj_sql_query = text(get_position_cmj_sql_string)
with engine.connect() as conn:
    position_cmj_df = pd.read_sql(get_position_cmj_sql_query, conn)
raw_position_cmj_df = position_cmj_df.sort_values(['date'])
## Storing position in variable to remove column
position_name = raw_position_cmj_df.tail(1)['Position'].iloc[0]
position_cmj_df = raw_position_cmj_df.groupby(['date', 'playerID']).max(numeric_only=True).reset_index()
position_cmj_df

,date,playerID,Jump Height
0,2025-01-11,278,0.3787
1,2025-01-11,279,0.4220
2,2025-01-11,280,0.2944
3,2025-01-13,4,0.3493
4,2025-01-13,30,0.3890
...,...,...,...
489,2025-10-06,85,0.4078
490,2025-10-06,114,0.3255
491,2025-10-06,278,0.4316
492,2025-10-06,279,0.3896


In [107]:
raw_position_cmj_df.groupby('date').mean(numeric_only=True).reset_index()[['date','Jump Height']]

,date,Jump Height
0,2025-01-11,0.358600
1,2025-01-13,0.315697
2,2025-01-20,0.312118
3,2025-01-27,0.325173
4,2025-02-03,0.335886
5,2025-02-10,0.339508
6,2025-02-17,0.348239
7,2025-02-24,0.351419
8,2025-03-04,0.333936
9,2025-03-06,0.339200


In [74]:
position_cmj_df = position_cmj_df.sort_values(['Jump Height']).tail(1)
highest_position_jump = position_cmj_df['Jump Height'].iloc[0]
highest_position_date = position_cmj_df['date'].iloc[0]
highest_position_player = position_cmj_df['playerID'].iloc[0]
highest_position_player = player_df.loc[player_df['playerid'] == highest_position_player]['Player'].iloc[0]
split_name = highest_position_player.split()
highest_position_player = " ".join([split_name[-1]] + split_name[:-1])
print(highest_position_player, highest_position_date, highest_position_jump)

split_name = highest_position_player.split()
highest_position_player = " ".join([split_name[-1]] + split_name[:-1])

Samson Okunlola 2025-06-06 0.5474


In [75]:
get_catapult_sql_string = f'''
select playerID, Date, [Total Player Load], [Maximum Velocity]
from catapult
where playerID = {playerid}
'''

get_catapult_sql_query = text(get_catapult_sql_string)
with engine.connect() as conn:
    catapult_df = pd.read_sql(get_catapult_sql_query, conn)

catapult_df.groupby(['Date']).agg({'Total Player Load' : 'sum', 'Maximum Velocity' : 'max'}).reset_index().sort_values(['Date'])

,Date,Total Player Load,Maximum Velocity
0,2023-01-17,370.90,15.73
1,2023-01-18,374.02,16.10
2,2023-01-20,386.55,15.19
3,2023-01-23,162.28,15.93
4,2023-01-24,480.64,14.91
...,...,...,...
405,2025-10-01,274.29,10.74
406,2025-10-02,265.05,12.46
407,2025-10-03,131.38,8.97
408,2025-10-04,256.77,11.11


In [76]:
get_body_weight_sql_string = f'''
select playerID, Date, Weight, [Ideal Weight], [BF%]
from body_weight
where playerID = {playerid}
'''

get_body_weight_sql_query = text(get_body_weight_sql_string)
with engine.connect() as conn:
    body_weight_df = pd.read_sql(get_body_weight_sql_query, conn)
body_weight_df = body_weight_df.sort_values(['Date'])
body_weight_df

,playerID,Date,Weight,Ideal Weight,BF%
0,86,2024-01-16,345,330.0,NaN
1,86,2024-01-17,342,330.0,NaN
2,86,2024-01-18,342,330.0,NaN
3,86,2024-01-19,343,330.0,NaN
4,86,2024-01-20,345,330.0,NaN
...,...,...,...,...,...
292,86,2025-09-30,325,327.5,20.7
293,86,2025-10-01,326,327.5,20.7
294,86,2025-10-02,327,327.5,20.7
295,86,2025-10-06,328,327.5,20.7


In [77]:
## Baseline Data
latest_weight = body_weight_df.tail(1)['Weight'].iloc[0]
latest_bf = body_weight_df.tail(1)['BF%'].iloc[0]

data = {'Metric' : ['Body Weight (lbs)', 'Body Fat (%)'], 'Value' : [latest_weight, latest_bf]}
baseline_df = pd.DataFrame(data)
baseline_df

,Metric,Value
0,Body Weight (lbs),328.0
1,Body Fat (%),20.7


In [87]:
get_internal_sql_string = f'''
select playerID, [Practice Date], SDNN, [Calories Burned]
from internalload2
where playerID = {playerid}
'''
get_internal_sql_query = text(get_internal_sql_string)
with engine.connect() as conn:
    internal_df = pd.read_sql(get_internal_sql_query, conn)
internal_df

,playerID,Practice Date,SDNN,Calories Burned
0,86,2025-06-10,114.382000,3323.280000
1,86,2025-06-12,114.415000,3519.330000
2,86,2025-06-13,116.408000,2247.930000
3,86,2025-06-23,110.476889,2923.908964
4,86,2025-06-24,111.609340,2956.617083
...,...,...,...,...
68,86,2025-09-29,177.517221,1266.090107
69,86,2025-09-30,156.151684,1932.843644
70,86,2025-10-01,158.346297,2167.843248
71,86,2025-10-02,129.974827,2175.684149


In [92]:
internal_df.mean(numeric_only=True)['Calories Burned']

np.float64(2092.1004231115307)

In [79]:
get_perch_sql_string = f'''
    select playerID, date, exerciseID, [Weight (lbs)], [Set Avg Peak Velocity (m/s)]
    from perch
    where playerID = {playerid}
    '''
get_perch_sql_query = text(get_perch_sql_string)
with engine.connect() as conn:
    perch_df = pd.read_sql(get_perch_sql_query, conn)

bench_data = perch_df.loc[perch_df['exerciseID'] == 1]

player_peak_bench = bench_data.sort_values(['Weight (lbs)']).tail(1)
player_peak_bench_rep = player_peak_bench['Weight (lbs)'].iloc[0]
player_peak_bench_date = player_peak_bench['date'].iloc[0]

bench_data = bench_data.groupby('date').mean(numeric_only=True).reset_index()

In [82]:
player_bench_max = perch_df.loc[perch_df['exerciseID'] == 1].sort_values(['Weight (lbs)']).tail(1)['Weight (lbs)'].iloc[0]
player_bench_max_date = perch_df.loc[perch_df['exerciseID'] == 1].sort_values(['Weight (lbs)']).tail(1)['date'].iloc[0]
player_squat_max = perch_df.loc[perch_df['exerciseID'] == 4].sort_values(['Weight (lbs)']).tail(1)['Weight (lbs)'].iloc[0]
player_squat_max_date = perch_df.loc[perch_df['exerciseID'] == 4].sort_values(['Weight (lbs)']).tail(1)['date'].iloc[0]
player_power_clean_max = perch_df.loc[perch_df['exerciseID'] == 9].sort_values(['Weight (lbs)']).tail(1)['Weight (lbs)'].iloc[0]
player_power_clean_max_date = perch_df.loc[perch_df['exerciseID'] == 9].sort_values(['Weight (lbs)']).tail(1)['date'].iloc[0]
print(player_power_clean_max, player_power_clean_max_date)

300 2025-07-14


In [83]:
perch_df.loc[perch_df['exerciseID'] == 9].sort_values(['Weight (lbs)']).tail(1)['Weight (lbs)'].iloc[0]

np.int64(300)